In [65]:
import pandas as pd
import numpy as np
import os

from sklearn.model_selection import train_test_split

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, MaxPooling1D, Dense, Dropout, Flatten

from sklearn.metrics import accuracy_score, classification_report

In [66]:
BASE_PATH = "../Data"

isot_df = pd.read_csv(os.path.join(BASE_PATH, "ISOT", "isot_cleaned.csv"))
welfake_df = pd.read_csv(os.path.join(BASE_PATH, "WELFake", "welfake_cleaned.csv"))

# Safety fix
isot_df["content"] = isot_df["content"].astype(str).fillna("")
welfake_df["content"] = welfake_df["content"].astype(str).fillna("")

print("ISOT:", isot_df.shape)
print("WELFake:", welfake_df.shape)

ISOT: (38827, 2)
WELFake: (62749, 2)


In [67]:
# ISOT
X_train_isot, X_test_isot, y_train_isot, y_test_isot = train_test_split(
    isot_df["content"], isot_df["label"], test_size=0.2, random_state=42
)

# WELFake
X_train_wel, X_test_wel, y_train_wel, y_test_wel = train_test_split(
    welfake_df["content"], welfake_df["label"], test_size=0.2, random_state=42
)

In [68]:
vocab_size = 20000
max_len = 500

tokenizer = Tokenizer(num_words=vocab_size)

# Fit on combined data (IMPORTANT)
tokenizer.fit_on_texts(pd.concat([X_train_isot, X_train_wel]))

# Convert text → sequences
X_train_isot_seq = tokenizer.texts_to_sequences(X_train_isot)
X_test_isot_seq = tokenizer.texts_to_sequences(X_test_isot)

X_train_wel_seq = tokenizer.texts_to_sequences(X_train_wel)
X_test_wel_seq = tokenizer.texts_to_sequences(X_test_wel)

# Padding
X_train_isot_pad = pad_sequences(X_train_isot_seq, maxlen=max_len)
X_test_isot_pad = pad_sequences(X_test_isot_seq, maxlen=max_len)

X_train_wel_pad = pad_sequences(X_train_wel_seq, maxlen=max_len)
X_test_wel_pad = pad_sequences(X_test_wel_seq, maxlen=max_len)

In [69]:
def create_cnn():
    model = Sequential([
        Embedding(input_dim=20000, output_dim=200, input_length=500),

        Conv1D(32, 5, activation='relu'),
        MaxPooling1D(2),

        Conv1D(64, 5, activation='relu'),
        MaxPooling1D(2),

        Conv1D(120, 3, activation='relu'),   # 🔥 extra layer
        MaxPooling1D(2),

        Flatten(),

        Dense(64, activation='relu'),
        Dropout(0.5),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(256, activation='relu'),
        Dropout(0.5),
        Dense(1024, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])

    model.compile(
        loss='binary_crossentropy',
        optimizer='adam',
        metrics=['accuracy']
    )

    return model

In [70]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

In [71]:
model_isot = create_cnn()

history_isot = model_isot.fit(
    X_train_isot_pad, y_train_isot,
    epochs=12,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop]
)

Epoch 1/12


C:\Users\Krish Kasnia\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


437/437 ━━━━━━━━━━━━━━━━━━━━ 21s 44ms/step - accuracy: 0.9056 - loss: 0.1934 - val_accuracy: 0.9858 - val_loss: 0.0418
Epoch 2/12
437/437 ━━━━━━━━━━━━━━━━━━━━ 18s 42ms/step - accuracy: 0.9932 - loss: 0.0260 - val_accuracy: 0.9939 - val_loss: 0.0220
Epoch 3/12
437/437 ━━━━━━━━━━━━━━━━━━━━ 18s 42ms/step - accuracy: 0.9977 - loss: 0.0088 - val_accuracy: 0.9952 - val_loss: 0.0180
Epoch 4/12
437/437 ━━━━━━━━━━━━━━━━━━━━ 18s 42ms/step - accuracy: 0.9986 - loss: 0.0068 - val_accuracy: 0.9945 - val_loss: 0.0293
Epoch 5/12
437/437 ━━━━━━━━━━━━━━━━━━━━ 18s 42ms/step - accuracy: 0.9989 - loss: 0.0048 - val_accuracy: 0.9961 - val_loss: 0.0177
Epoch 6/12
437/437 ━━━━━━━━━━━━━━━━━━━━ 18s 42ms/step - accuracy: 0.9976 - loss: 0.0096 - val_accuracy: 0.9949 - val_loss: 0.0200
Epoch 7/12
437/437 ━━━━━━━━━━━━━━━━━━━━ 18s 42ms/step - accuracy: 0.9986 - loss: 0.0093 - val_accuracy: 0.9952 - val_loss: 0.0175
Epoch 8/12
437/437 ━━━━━━━━━━━━━━━━━━━━ 19s 43ms/step - accuracy: 0.9986 - loss: 0.0073 - val_accurac

In [72]:
from sklearn.metrics import accuracy_score

y_pred_isot = (model_isot.predict(X_test_isot_pad) > 0.5).astype("int32")

isot_acc = accuracy_score(y_test_isot, y_pred_isot)

print("ISOT Accuracy:", isot_acc)

243/243 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step
ISOT Accuracy: 0.9927890806077775


In [73]:
model_wel = create_cnn()

history_wel = model_wel.fit(
    X_train_wel_pad, y_train_wel,
    epochs=12,
    batch_size=64,
    validation_split=0.1,
    callbacks=[early_stop]
)

Epoch 1/12


C:\Users\Krish Kasnia\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


706/706 ━━━━━━━━━━━━━━━━━━━━ 33s 44ms/step - accuracy: 0.9057 - loss: 0.2298 - val_accuracy: 0.9669 - val_loss: 0.1104
Epoch 2/12
706/706 ━━━━━━━━━━━━━━━━━━━━ 30s 43ms/step - accuracy: 0.9789 - loss: 0.0662 - val_accuracy: 0.9695 - val_loss: 0.0844


In [74]:
y_pred_wel = (model_wel.predict(X_test_wel_pad) > 0.5).astype("int32")

wel_acc = accuracy_score(y_test_wel, y_pred_wel)

print("WELFake Accuracy:", wel_acc)

393/393 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step
WELFake Accuracy: 0.9650996015936255


In [75]:
import pandas as pd

cnn_results = pd.DataFrame({
    "Dataset": ["ISOT", "WELFake"],
    "Model": ["CNN", "CNN"],
    "Accuracy": [isot_acc, wel_acc]
})

cnn_results["Accuracy (%)"] = (cnn_results["Accuracy"] * 100).round(2)

print(cnn_results)

   Dataset Model  Accuracy  Accuracy (%)
0     ISOT   CNN  0.992789         99.28
1  WELFake   CNN  0.965100         96.51


In [76]:
# import os

# os.makedirs("../models", exist_ok=True)

# model_isot.save("../models/cnn_isot.h5")
# model_wel.save("../models/cnn_welfake.h5")

# print("✅ CNN Models Saved")